# Reproducing Daume et al. (2024) from dandiset 001187

Paper: Daume, Kaminski, Salimpour, Gomez Palacio Schjetnan, Anderson, Valiante, Mamelak and
Rutishauser (2024), "Persistent activity during working memory maintenance predicts long-term
memory formation in the human hippocampus", Neuron 112, 3957-3968,
doi:10.1016/j.neuron.2024.09.013.

Dataset: DANDI:001187, version 0.240904.2347 (doi:10.48324/dandi.001187/0.240904.2347). This is
the version the paper cites (Data and Code Availability; key resources table), and it is the
only published version; the draft has the same 46 assets and size.

Claims: `claims/001187/10.1016-j.neuron.2024.09.013.yaml`, written from the accepted author
manuscript in PubMed Central (PMC11624075) with its supplement (Figures S1 to S4, Table S1),
and checked by a second reader. Method details the claims file lacks are taken from the paper
or its code (github.com/rutishauserlab/SBCAT-NO-release-NWB, commit 6d07204) and cited where
used, here and in `sbcat.py`.

| Claim | Tested or blocked |
|---|---|
| sessions-and-neurons, table-s1-counts | tested |
| behaviour-overall-accuracy, fig1c to fig1g, ltm-rt-confidence | tested |
| category-neuron-counts, fig2b, fig2c, fig2e, fig2f | tested |
| fig3a to fig3h (nine claims) | tested |
| ms-neuron-counts, fig4b, fig4c, fig4e, fig4f | tested |
| fig4d-category-no-memory-signal | blocked: the new LTM pictures have no category in the dataset |

Every number below is computed from the streamed files. Seeds are fixed, one generator per
claim section.

In [ ]:
import time
import numpy as np
import pandas as pd
import sbcat

T0 = time.time()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)
print("dandiset", sbcat.DANDISET, "version", sbcat.VERSION)

## Data check

Every NWB file of the pinned version is streamed; the trial tables, the raw event times, the
electrode locations and the spike times are read (not the waveforms or the stimulus images).

In [ ]:
S = sbcat.load_all()
print(len(S), "sessions loaded in", round(time.time() - T0), "s")

In [ ]:
rows = []
for si, s in enumerate(S):
    ltm, wm = s["ltm"], s["wm"]
    conf = ltm.confidence.values
    resp_old = conf >= 4
    rows.append(dict(
        session=s["identifier"], subject=s["subject_id"], ses=s["session_id"], age=s["age"],
        sex=s["sex"], institution=s["institution"], units=len(s["spikes"]),
        hipp=int(sum(sbcat.area_of(l) == "hippocampus" for l in s["units"].location)),
        amyg=int(sum(sbcat.area_of(l) == "amygdala" for l in s["units"].location)),
        wm_trials=len(wm), load1=int((wm.loads == 1).sum()), load3=int((wm.loads == 3).sum()),
        ltm_trials=len(ltm), ltm_old=int((ltm.new_old == 1).sum()),
        conf_values="".join(str(v) for v in np.unique(conf)),
        acc_mismatch=int(np.sum((resp_old == (ltm.new_old.values == 1)).astype(int)
                                != ltm.response_accuracy.values)),
        unsorted_units=len(s["unsorted_units"]),
        variant=s["notes"].split("Experiment variant: ")[1][0]))
SES = pd.DataFrame(rows)
print(SES.to_string())
print("\nsessions", len(SES), "subjects", SES.subject.nunique(), "units", SES.units.sum(),
      "hippocampus", SES.hipp.sum(), "amygdala", SES.amyg.sum())
print("locations:", sorted({l for s in S for l in s["units"].location}))
print("institutions in the files:", SES.institution.unique().tolist())
print("sessions by identifier suffix (site):",
      pd.Series([s.split("_")[-1].lstrip("P0123456789") for s in SES.session]).value_counts().to_dict())
print("sessions with fewer than 400 LTM trials:",
      SES.loc[SES.ltm_trials < 400, ["session", "ltm_trials", "ltm_old"]].values.tolist())
print("sessions with fewer than 140 WM trials:",
      SES.loc[SES.wm_trials < 140, ["session", "wm_trials"]].values.tolist())
print("confidence values outside 1 to 6:",
      [(s["identifier"], int(v), int(np.sum(s["ltm"].confidence == v)))
       for s in S for v in np.unique(s["ltm"].confidence) if v < 1 or v > 6])
print("trials whose accuracy disagrees with 'old' = confidence 4 to 6:", SES.acc_mismatch.sum())
print("units with unsorted spike times:", SES.unsorted_units.sum(),
      [(s["identifier"], s["unsorted_units"]) for s in S if s["unsorted_units"]])
allwm = pd.concat([s["wm"] for s in S])
allltm = pd.concat([s["ltm"] for s in S])
print("\nWM response time (s): %.3f to %.3f, median %.3f" % (allwm.response_time.min(),
      allwm.response_time.max(), allwm.response_time.median()))
print("LTM response time (s): %.3f to %.3f, median %.3f" % (allltm.response_time.min(),
      allltm.response_time.max(), allltm.response_time.median()))
print("maintenance length (probe - maintenance onset, s): %.3f to %.3f" % (
      (allwm.timestamps_Probe - allwm.timestamps_Maintenance).min(),
      (allwm.timestamps_Probe - allwm.timestamps_Maintenance).max()))
enc = allwm.timestamps_Encoding1_end - allwm.timestamps_Encoding1
print("encoding picture 1 duration (s): %.3f to %.3f" % (enc.min(), enc.max()))
print("maintenance onset - end of last picture (s): max abs %.4f" % np.max(np.abs(
      allwm.timestamps_Maintenance - np.where(allwm.loads == 1, allwm.timestamps_Encoding1_end,
                                              allwm.timestamps_Encoding3_end))))
print("picture categories (leading digit):", sorted(np.unique(allwm.PicIDs_Encoding1 // 100)))
print("LTM load values:", sorted(allltm.load.unique()), " num_times_shown:",
      sorted(allltm.num_times_shown.unique()))
print("load of new pictures:", sorted(allltm.load[allltm.new_old == 0].unique()),
      " num_times_shown of new:", sorted(allltm.num_times_shown[allltm.new_old == 0].unique()))

Event clock. The trial tables and the spikes are checked against each other below, by the
response of the units to picture onset (the category selection decodes the picture
category from spikes aligned to the WM table's picture onsets; shifting the onsets removes
it), see the cell after the category selection.

## Units and the rate filter (claims U)

In [ ]:
rows = []
for si, s in enumerate(S):
    for ui, loc in enumerate(s["units"].location):
        wmr, ltmr = sbcat.global_rates(s["spikes"][ui], s["raw_WM"], s["raw_LTM"])
        rows.append(dict(si=si, ui=ui, session=s["identifier"], patient=s["subject_id"],
                         location=loc, area=sbcat.area_of(loc), rate_wm=wmr, rate_ltm=ltmr))
UN = pd.DataFrame(rows)
UN["keep"] = (UN.rate_wm >= 0.1) & (UN.rate_ltm >= 0.1)
print("global WM rate (Hz): %.3f to %.2f; LTM rate: %.3f to %.2f" % (
    UN.rate_wm.min(), UN.rate_wm.max(), UN.rate_ltm.min(), UN.rate_ltm.max()))
print(UN.groupby("area").keep.agg(["size", "sum"]))
K = UN[UN.keep].reset_index(drop=True)
print("kept", len(K), "removed", int((~UN.keep).sum()))

Duplicate recordings. Two sessions could hold the same recording deposited twice. For every
pair of sessions, the trial tables and the spike trains are compared.

In [ ]:
dups = []
sig = [(s["identifier"], s["wm"].timestamps_Encoding1.values, s["wm"].PicIDs_Encoding1.values,
        [len(x) for x in s["spikes"]]) for s in S]
for i in range(len(S)):
    for j in range(i + 1, len(S)):
        same_pics = np.array_equal(sig[i][2], sig[j][2])
        same_times = len(sig[i][1]) == len(sig[j][1]) and np.allclose(sig[i][1] - sig[i][1][0],
                                                                      sig[j][1] - sig[j][1][0])
        common_counts = len(set(sig[i][3]) & set(sig[j][3]) - {0})
        if same_times or common_counts > 2:
            dups.append((sig[i][0], sig[j][0], same_pics, same_times, common_counts))
print("session pairs with identical WM timing or more than 2 units of identical spike counts:", dups)
same_order = pd.Series([tuple(s["wm"].PicIDs_Encoding1) for s in S]).duplicated(keep=False)
print("sessions sharing the same picture sequence (expected for one task variant):",
      int(same_order.sum()))

Checks of the description against the data, from the outputs above. The confidence column
is described as 1 to 6 with no coding; the accuracy column agrees in every trial with 1 to 3
meaning "new" and 4 to 6 "old" (shown). One trial has the value 7 (shown); it counts as "old"
under that coding, which agrees with its accuracy. The LTM table's load is 0 for new
pictures and num_times_shown 0 for new pictures (shown). Response times are in seconds
(0.135 to 56 s in WM; up to 130 s in LTM, where the picture stays until a response). The
maintenance period lasts 2.53 to 3.20 s (the Methods say 2.55 to 2.85 s) and starts at the end
of the last picture. Picture 1 lasts 1.57 to 2.04 s where the Methods say 2 s. The files
name Cedars-Sinai for every session, which the identifiers contradict (shown). No session
pair shares its trial timing or spike counts, so no recording is deposited twice; the 45
sessions sharing one picture sequence are the same task version. The same units are taken
to be recorded in both tasks of a session (one sorting per session, as the Methods say). The
clock check after the category selection supports a shared clock of spikes and trial tables
where sessions have enough units.

## Claim sessions-and-neurons

> After excluding two sessions, 46 sessions are analysed, in which 883 single units pass the
> 0.1 Hz rate filter, 351 in the hippocampus and 532 in the amygdala, of 950 isolated.

In [ ]:
S1 = sbcat.table_s1()
print("sessions in the dandiset:", len(S), "(Table S1 lists", len(S1), "kept sessions)")
print("isolated units:", len(UN), " Table S1 recorded total:", S1.h_rec.sum() + S1.a_rec.sum())
print("after the rate filter: hippocampus", int(K.area.eq("hippocampus").sum()),
      " amygdala", int(K.area.eq("amygdala").sum()))
print("removed by the filter:", int((~UN.keep).sum()), "(%.1f%%)" % (100 * (~UN.keep).mean()))
print("Table S1 sums: hippocampus", S1.h_rec.sum(), " amygdala", S1.a_rec.sum())

**Result: reproduced.** The dandiset holds 46 sessions and 950 units; the rate filter removes 67 (7.1%) and leaves 351 hippocampal and 532 amygdala units, as the paper states. The recorded counts of Table S1 sum to 973.

## Claim table-s1-counts

> Per session, the numbers of recorded units in the hippocampus and the amygdala are those of
> Table S1.

Sessions are matched by the label in the file identifier (P54CS with session 2 is P54cs_2).

In [ ]:
SES["label"] = [sbcat.s1_label(x) for x in SES.session]
kept_counts = K.groupby(["session", "area"]).size().unstack(fill_value=0)
SES["hipp_kept"] = SES.session.map(kept_counts.get("hippocampus", {})).fillna(0).astype(int)
SES["amyg_kept"] = SES.session.map(kept_counts.get("amygdala", {})).fillna(0).astype(int)
SES["wm_acc"] = [100 * s["wm"].response_accuracy.mean() for s in S]
SES["ltm_acc"] = [100 * s["ltm"].response_accuracy.mean() for s in S]
TS1 = SES.merge(S1, on="label", how="outer", indicator=True)
print("matched:", (TS1._merge == "both").sum(), " only in dandiset:",
      TS1.label[TS1._merge == "left_only"].tolist(), " only in Table S1:",
      TS1.label[TS1._merge == "right_only"].tolist())
TS1 = TS1[TS1._merge == "both"].copy()
TS1["match_before"] = (TS1.hipp == TS1.h_rec) & (TS1.amyg == TS1.a_rec)
TS1["match_after"] = (TS1.hipp_kept == TS1.h_rec) & (TS1.amyg_kept == TS1.a_rec)
TS1["match_either"] = TS1.match_before | TS1.match_after
print("sessions matching before the filter:", int(TS1.match_before.sum()), "after:",
      int(TS1.match_after.sum()), "under either:", int(TS1.match_either.sum()), "of", len(TS1))
print(TS1.loc[~TS1.match_either, ["label", "session", "hipp", "hipp_kept", "h_rec", "amyg",
                                  "amyg_kept", "a_rec"]].to_string())
dwm, dltm = TS1.wm_acc_x - TS1.wm_acc_y, TS1.ltm_acc_x - TS1.ltm_acc_y
print("\naccuracy against Table S1 (a check of the session match): max |WM diff| %.2f, "
      "max |LTM diff| %.2f points" % (dwm.abs().max(), dltm.abs().max()))
print(TS1.loc[(dwm.abs() > 0.15) | (dltm.abs() > 0.15),
              ["label", "wm_acc_x", "wm_acc_y", "ltm_acc_x", "ltm_acc_y", "ltm_trials"]].round(2).to_string())

**Result: partially reproduced.** 42 of the 46 sessions match Table S1's recorded counts before the rate filter (21 after it). Four differ: P1809jh (1 hippocampal unit against 6), P61cs_2 (1 against 2), P68cs (7 amygdala units against 17) and P82cs (17 against 24); the 23 missing units are the difference between Table S1's 973 and the 950 of the text. The session match is confirmed by the accuracies, which differ from Table S1 by at most 0.18 points (WM) and 1.52 points (LTM, in sessions with incomplete LTM tables).

## Behaviour

Per session measures (claims S, L). High confidence is a rating of 1 or 6 (sure), low
confidence 2 to 5. Sessions "with confidence ratings" leave out the one session whose ratings
are only 1 and 6 (printed below), the patient the paper says "did not use confidence ratings"
(Results, paragraph 2); the paper's code drops it the same way, as its low-confidence accuracy is
undefined (plot_behavior.m).

In [ ]:
from scipy.stats import norm
rows = []
for s in S:
    wm, ltm = s["wm"], s["ltm"]
    conf = ltm.confidence.values
    old = ltm.new_old.values == 1
    said_old = conf >= 4
    acc = ltm.response_accuracy.values.astype(float)
    hi = (conf == 1) | (conf == 6)
    unsure = (conf == 2) | (conf == 5)
    guess = (conf == 3) | (conf == 4)
    lo = ~hi
    rt = ltm.response_time.values
    rt5 = np.where(rt > 5, np.nan, rt)
    wrt = wm.response_time.values
    wrt5 = np.where(wrt > 5, np.nan, wrt)
    hit, fa = said_old[old].mean(), said_old[~old].mean()
    nh, nf = old.sum(), (~old).sum()
    hit_c = min(max(hit, 1 / (2 * nh)), 1 - 1 / (2 * nh))
    fa_c = min(max(fa, 1 / (2 * nf)), 1 - 1 / (2 * nf))
    L = ltm.load.values
    nts = ltm.num_times_shown.values
    rows.append(dict(
        session=s["identifier"], wm_acc=wm.response_accuracy.mean(), ltm_acc=acc.mean(),
        hit=hit, fa=fa, dprime=norm.ppf(hit_c) - norm.ppf(fa_c), rate_corrected=(hit != hit_c) or (fa != fa_c),
        wm_acc_l1=wm.response_accuracy[wm.loads == 1].mean(), wm_acc_l3=wm.response_accuracy[wm.loads == 3].mean(),
        wm_rt_l1=np.nanmean(wrt5[wm.loads == 1]), wm_rt_l3=np.nanmean(wrt5[wm.loads == 3]),
        wm_rt_l1_all=np.mean(wrt[wm.loads == 1]), wm_rt_l3_all=np.mean(wrt[wm.loads == 3]),
        n_hi=hi.sum(), n_unsure=unsure.sum(), n_guess=guess.sum(),
        acc_hi=acc[hi].mean() if hi.any() else np.nan,
        acc_lo_pooled=acc[lo].mean() if lo.any() else np.nan,
        acc_lo_avg=np.nanmean([acc[unsure].mean() if unsure.any() else np.nan,
                               acc[guess].mean() if guess.any() else np.nan]) if lo.any() else np.nan,
        rt_hi_all=rt[hi].mean(), rt_lo_pooled_all=rt[lo].mean() if lo.any() else np.nan,
        rt_lo_avg_all=np.nanmean([rt[unsure].mean() if unsure.any() else np.nan,
                                  rt[guess].mean() if guess.any() else np.nan]) if lo.any() else np.nan,
        rt_hi_5=np.nanmean(rt5[hi]), rt_lo_pooled_5=np.nanmean(rt5[lo]) if lo.any() else np.nan,
        rt_lo_avg_5=np.nanmean([np.nanmean(rt5[unsure]) if unsure.any() else np.nan,
                                np.nanmean(rt5[guess]) if guess.any() else np.nan]) if lo.any() else np.nan,
        hit_l1=said_old[old & (L == 1)].mean(), hit_l3=said_old[old & (L == 3)].mean(),
        hit_twice=said_old[old & (nts == 2)].mean(), hit_once=said_old[old & (nts == 1)].mean()))
BEH = pd.DataFrame(rows)
noconf = BEH.session[BEH.n_hi == [len(s["ltm"]) for s in S]].tolist()
print("sessions whose ratings are all high confidence:", noconf)
BEHC = BEH[~BEH.session.isin(noconf)]
print("sessions with confidence ratings:", len(BEHC))
print("sessions without guessing ratings:", BEHC.session[BEHC.n_guess == 0].tolist())
print(BEH.describe().T[["count", "mean", "std", "min", "max"]].round(3).to_string())

## Claim behaviour-overall-accuracy

> Patients answered 93.3% of WM trials and 70.3% of LTM trials correctly (d' 1.19), both above
> the 50% chance level.

In [ ]:
rng = np.random.default_rng(101)
for col, exp, tol in [("wm_acc", 93.3, 1.1), ("ltm_acc", 70.3, 1.4)]:
    v = 100 * BEH[col]
    t, p, df = sbcat.perm_ttest_1samp(v, 50, rng)
    print(f"{col}: mean {v.mean():.2f} % (SD {v.std(ddof=1):.2f}), expected {exp} +- {tol}, "
          f"within: {abs(v.mean() - exp) <= tol}; against 50%: t({df}) = {t:.2f}, perm p = {p:.3g} (below 0.0001: {p < 1e-4})")
print(f"d': mean {BEH.dprime.mean():.3f} (SD {BEH.dprime.std(ddof=1):.3f}), expected 1.19 +- 0.09, "
      f"within: {abs(BEH.dprime.mean() - 1.19) <= 0.09}; sessions with a hit or FA rate of 0 or 1 "
      f"(1/(2N) applied): {int(BEH.rate_corrected.sum())}")
print("smallest p a 10,000-permutation test gives here: %.3g" % (2 * 0.5 / 10001))

**Result: reproduced.** Mean WM accuracy 93.33% (SD 7.60), LTM accuracy 70.32% (SD 9.89) and d' 1.189 (SD 0.620) are within tolerance, and both accuracies exceed 50% at the smallest p a 10,000-permutation test gives (below 0.0001).

## Claim fig1c-wm-accuracy-load

> WM accuracy is higher in load 1 than in load 3 trials.

In [ ]:
rng = np.random.default_rng(102)
t, p, df = sbcat.perm_ttest_paired(BEH.wm_acc_l1, BEH.wm_acc_l3, rng)
print(f"load 1 {100*BEH.wm_acc_l1.mean():.2f} %, load 3 {100*BEH.wm_acc_l3.mean():.2f} %; "
      f"t({df}) = {t:.2f} (paper 4.89), perm p = {p:.3g} (below 0.0001: {p < 1e-4}); sessions with load 1 higher: "
      f"{int((BEH.wm_acc_l1 > BEH.wm_acc_l3).sum())}, equal {int((BEH.wm_acc_l1 == BEH.wm_acc_l3).sum())}")

**Result: reproduced.** Load 1 accuracy 95.09% against 91.58% for load 3, t(45) = 4.91 (paper 4.89), permutation p below 0.0001.

## Claim fig1d-wm-rt-load

> WM responses are faster in load 1 than in load 3 trials.

In [ ]:
rng = np.random.default_rng(103)
for lab, a, b in [("RT > 5 s missing (main)", "wm_rt_l1", "wm_rt_l3"),
                  ("all RTs (variant)", "wm_rt_l1_all", "wm_rt_l3_all")]:
    t, p, df = sbcat.perm_ttest_paired(BEH[a], BEH[b], rng)
    print(f"{lab}: load 1 {BEH[a].mean():.3f} s, load 3 {BEH[b].mean():.3f} s; t({df}) = {t:.2f} "
          f"(paper -4.29), perm p = {p:.3g} (below 0.0001: {p < 1e-4})")
print("WM trials with RT > 5 s:", int(sum((s["wm"].response_time > 5).sum() for s in S)))

**Result: partially reproduced.** With response times above 5 s set to missing, load 1 is faster (1.393 against 1.525 s), t(45) = -4.28 as in the paper (-4.29), but the permutation p is 0.0003, not below 0.0001 as the criterion requires for a full reproduction. With all response times (variant), there is no difference (t = 0.20).

## Claim fig1e-ltm-confidence-accuracy

> LTM recognition is more accurate in high-confidence than in low-confidence trials.

In [ ]:
rng = np.random.default_rng(104)
for lab, col in [("(a) unsure and guessing pooled", "acc_lo_pooled"),
                 ("(b) mean of unsure and guessing accuracy", "acc_lo_avg")]:
    t, p, df = sbcat.perm_ttest_paired(BEHC.acc_hi, BEHC[col], rng)
    print(f"{lab}: high {100*BEHC.acc_hi.mean():.2f} %, low {100*BEHC[col].mean():.2f} %; "
          f"t({df}) = {t:.2f} (paper 8.66), perm p = {p:.3g} (below 0.0001: {p < 1e-4})")

**Result: reproduced.** High-confidence accuracy 77.31% against 61.40% (pooled low, t(44) = 8.51) or 62.48% (averaged, t(44) = 7.34), both with permutation p below 0.0001, in 45 sessions.

## Claim ltm-rt-confidence

> LTM responses are faster with high confidence (mean 1.83 s) than with low confidence (2.62 s).

In [ ]:
rng = np.random.default_rng(105)
for rtr, suf in [("(a) all RTs", "all"), ("(b) RT > 5 s missing", "5")]:
    for lr, low in [("pooled", "pooled"), ("averaged", "avg")]:
        h, l = BEHC[f"rt_hi_{suf}"], BEHC[f"rt_lo_{low}_{suf}"]
        t, p, df = sbcat.perm_ttest_paired(h, l, rng)
        print(f"{rtr}, low {lr}: high {h.mean():.3f} s (SD {h.std(ddof=1):.2f}; 1.83 +- 0.085: "
              f"{abs(h.mean()-1.83) <= 0.085}), low {l.mean():.3f} s (SD {l.std(ddof=1):.2f}; 2.62 +- 0.098: "
              f"{abs(l.mean()-2.62) <= 0.098}); t({df}) = {t:.2f} (paper -11.13), perm p = {p:.3g} (below 0.0001: {p < 1e-4})")

**Result: reproduced.** With response times above 5 s set to missing (reading b), the means are 1.838 s (high) and 2.618 s (pooled low) or 2.659 s (averaged low), within tolerance, with t(44) = -11.37 or -11.22 (paper -11.13) and p below 0.0001. With all response times (reading a), the means are 2.205 and 3.565 s, outside tolerance, though high is still faster (p below 0.0001).

## Claim fig1f-ltm-load

> Pictures held in load 1 WM trials are later recognized better than pictures held in load 3
> trials.

In [ ]:
rng = np.random.default_rng(106)
t, p, df = sbcat.perm_ttest_paired(BEH.hit_l1, BEH.hit_l3, rng)
print(f"hit rate load 1 {100*BEH.hit_l1.mean():.2f} %, load 3 {100*BEH.hit_l3.mean():.2f} %; "
      f"t({df}) = {t:.2f} (paper 2.42), perm p = {p:.4f} (paper 0.018)")

**Result: reproduced.** Hit rate 68.59% for load 1 pictures against 65.77% for load 3, t(45) = 2.65, p = 0.0097 (paper t = 2.42, p = 0.018).

## Claim fig1g-ltm-probe

> Pictures that were also used as WM probes, and so shown twice, are later recognized better
> than pictures shown once.

In [ ]:
rng = np.random.default_rng(107)
t, p, df = sbcat.perm_ttest_paired(BEH.hit_twice, BEH.hit_once, rng)
print(f"hit rate shown twice {100*BEH.hit_twice.mean():.2f} %, once {100*BEH.hit_once.mean():.2f} %; "
      f"t({df}) = {t:.2f} (paper 7.25), perm p = {p:.3g} (below 0.0001: {p < 1e-4})")

**Result: reproduced.** Hit rate 72.10% for pictures shown twice against 63.99% for pictures shown once, t(45) = 6.80, p below 0.0001.

## Category and memory-selective neurons (selections C and M)

Each kept unit is tested as the paper's code does (NWB_calcCatSelective_SB.m,
NWB_calcMemSelective_SB.m), with 10,000 permutations. Seeds are standard choices.

In [ ]:
rng = np.random.default_rng(201)
t0 = time.time()
cat_res, mem_res = [], []
for si, s in enumerate(S):
    times, cats = sbcat.category_rates(s)
    for _, u in K[K.si == si].iterrows():
        sp = s["spikes"][u.ui]
        rate = sbcat.count(sp, times, 0.2, 1.2) / 1.0
        cat_res.append(sbcat.category_test(rate, cats, rng))
rng = np.random.default_rng(202)
for si, s in enumerate(S):
    for _, u in K[K.si == si].iterrows():
        mem_res.append(sbcat.memory_test(s, s["spikes"][u.ui], rng))
K["cat"] = [r[0] for r in cat_res]
K["pref"] = [int(r[1]) for r in cat_res]
K["p_anova"] = [r[2] for r in cat_res]
K["p_cat_t"] = [r[3] for r in cat_res]
K["p_ms"] = [r[0] for r in mem_res]
K["ms"] = K.p_ms < 0.05
K["ms_type"] = np.where(~K.ms, "", np.where([r[1] > 0 for r in mem_res], "FS", "NS"))
print("selection time %.0f s" % (time.time() - t0))
print(K.groupby("area")[["cat", "ms"]].sum())
print("units whose ANOVA was undefined (no spikes in any window):", int(K.p_anova.isna().sum()))

Clock check. If the trial tables share the spikes' clock, the category of the picture on the
screen is decodable from spikes aligned to the table's onsets and not from spikes aligned to
onsets shifted away from the picture. The same holds for old against new in the LTM task. Per
session, the fraction of units with a parametric one-way ANOVA (WM categories) or t-test (LTM
old against new, correct trials) at p < 0.05, for the onsets as given and shifted by -5 s
(into the preceding trial; a contrast between labels, not an offset).

In [ ]:
from scipy import stats as st
rows = []
for si, s in enumerate(S):
    times, cats = sbcat.category_rates(s)
    ltm = s["ltm"]
    ok = ltm.response_accuracy.values == 1
    tl, old = ltm.timestamps_PicOnset.values[ok], ltm.new_old.values[ok] == 1
    for shift in (0.0, -5.0):
        nw = nl = 0
        for _, u in K[K.si == si].iterrows():
            sp = s["spikes"][u.ui]
            r = sbcat.count(sp, times + shift, 0.2, 1.2)
            if r.std() > 0 and st.f_oneway(*[r[cats == c] for c in np.unique(cats)]).pvalue < 0.05:
                nw += 1
            c = sbcat.count(sp, tl + shift, 0.2, 1.2)
            if c.std() > 0 and st.ttest_ind(c[old], c[~old]).pvalue < 0.05:
                nl += 1
        rows.append(dict(session=s["identifier"], shift=shift, units=int((K.si == si).sum()),
                         wm_sig=nw, ltm_sig=nl))
CLK = pd.DataFrame(rows).pivot(index="session", columns="shift", values=["units", "wm_sig", "ltm_sig"])
print(CLK.to_string())
tot = pd.DataFrame(rows).groupby("shift")[["units", "wm_sig", "ltm_sig"]].sum()
print(tot)
print("sessions where the true WM onsets give no more significant units than the shifted ones:",
      CLK.index[CLK[("wm_sig", 0.0)] <= CLK[("wm_sig", -5.0)]].tolist())
print("sessions where the true LTM onsets give no more significant units than the shifted ones:",
      CLK.index[CLK[("ltm_sig", 0.0)] <= CLK[("ltm_sig", -5.0)]].tolist())

## Claim category-neuron-counts

> 104 hippocampal units (29.65%) and 220 amygdala units (41.35%) are category neurons.

In [ ]:
for area, exp in [("hippocampus", 104), ("amygdala", 220)]:
    k = K[K.area == area]
    n = int(k.cat.sum())
    print(f"{area}: {n} of {len(k)} ({100*n/len(k):.2f} %), expected {exp} +- {0.1*exp:.1f}: "
          f"{abs(n-exp) <= 0.1*exp}")
    print("   preferred categories:", k[k.cat].pref.value_counts().sort_index().to_dict())
for area in ("hippocampus", "amygdala"):
    k = K[K.area == area]
    print(area, "with the second test at a one-sided 0.05 read as two-sided 0.05 (variant):",
          int(((k.p_anova < 0.05) & (k.p_cat_t < 0.05)).sum()))
byp = K[K.cat].groupby(["area", "patient"]).size().unstack(0, fill_value=0)
print("category neurons by patient (hippocampus, amygdala):")
print(byp.T.to_string())

**Result: reproduced.** 102 hippocampal (29.06%) and 222 amygdala (41.73%) category neurons, within 10% of 104 and 220.

## Tables of category neurons

For each category neuron, one row per encoding picture of every WM trial with its spike
counts in the maintenance (0 to 2.5 s after maintenance onset), baseline (-0.9 to -0.3 s
before picture 1) and picture 1 (0 to 2 s) windows, and, for pictures shown in the LTM test,
that trial's accuracy and confidence (sbcat.picture_table, sbcat.add_counts).

Outliers for the GLMs of Figures 2 and 3 (claims O, as amended after a reader's answer in
`questions/001187.yaml`): O-G, each neuron's mean normalized rate over all rows of its table
before trial selection, as NWB_computeGLM.m computes it, the neuron removed when its mean lies
outside the mean +- 3 SD of the neuron means (Methods, Statistics, "differed ±3 SD from the
mean"); O-C, the code's rule (above 3 SD of the neuron means, no centring), as a variant; and
no removal. Neurons with a zero baseline are dropped under all three.

In [ ]:
PT = {si: sbcat.picture_table(S[si]) for si in sorted(K.si.unique())}
CN = K[K.cat].copy()
CN["neuron"] = [f"{a}_{b}" for a, b in zip(CN.si, CN.ui)]
NPT = {}
for _, u in CN.iterrows():
    pt = sbcat.add_counts(PT[u.si], S[u.si]["spikes"][u.ui])
    pt["pref"] = (pt.cat == u.pref).astype(int)
    pt["neuron"], pt["patient"], pt["area"] = u.neuron, u.patient, u.area
    NPT[u.neuron] = pt
print("category neurons:", len(NPT), " picture rows per neuron:",
      sorted({len(p) for p in NPT.values()}))
print("pictures shown in the LTM test per session: %d to %d" % (
    min(p.in_ltm.sum() for p in PT.values()), max(p.in_ltm.sum() for p in PT.values())))


def trial_rows(pt):
    """One row per WM trial; pref is true when any encoding picture is of the preferred
    category."""
    return pt.groupby("trial").agg(r_maint=("r_maint", "first"), r_bsl=("r_bsl", "first"),
                                   c_maint=("c_maint", "first"), c_bsl=("c_bsl", "first"),
                                   r_enc1=("r_enc1", "first"), acc=("acc_WM", "first"),
                                   load=("load", "first"), pref=("pref", "max"),
                                   neuron=("neuron", "first"), patient=("patient", "first"),
                                   area=("area", "first")).reset_index()


def code_rows(pt):
    """The rows of the paper's code (NWB_computeGLM.m): one per encoding picture later shown
    in the LTM test, pref per picture."""
    return pt[pt.in_ltm].rename(columns={"acc_WM": "acc"}).reset_index(drop=True)


def wm_table(neurons, rows, base, value="r_maint"):
    """Normalized rates (claims B) for the given neurons. rows 'trial' or 'code'; base 'B1'
    (mean baseline over all WM trials of the neuron) or 'B2' (over the rows of the code's
    table). Neurons with a zero baseline are dropped (their values are infinite)."""
    out = []
    for n in neurons:
        pt = NPT[n]
        tr = trial_rows(pt)
        d = tr if rows == "trial" else code_rows(pt)
        b = tr.r_bsl.mean() if base == "B1" else code_rows(pt).r_bsl.mean()
        if b == 0:
            continue
        d = d.copy()
        d["FR"] = sbcat.normalize(d[value], b)
        out.append(d)
    return pd.concat(out, ignore_index=True)


def fit_readings(neurons, select, formula, readings, value="r_maint"):
    """Fit the GLM under each reading; readings are (rows, base) pairs. Outliers by O-G
    (main, claims O) and O-C (variant), both on the neuron means over all rows of the table
    before selection, as NWB_computeGLM.m takes them, and none."""
    res = []
    for rows, base in readings:
        full = wm_table(neurons, rows, base, value)
        d = select(full)
        for orule in ("O-G", "O-C", "none"):
            bad = (sbcat.og_remove(full) if orule == "O-G" else
                   sbcat.oc_remove(full) if orule == "O-C" else set())
            dd = d[~d.neuron.isin(bad)]
            m = sbcat.mixedlm(dd, formula)
            for term, r in m.iterrows():
                res.append(dict(rows=rows, base=base, outliers=orule, term=term, beta=r.beta,
                                SE=r.SE, p=r.p, neurons=m.attrs["neurons"], n=m.attrs["n"],
                                removed=len(bad), converged=m.attrs["converged"]))
    return pd.DataFrame(res)


def show(res, terms=None):
    r = res if terms is None else res[res.term.isin(terms)]
    print(r.round({"beta": 2, "SE": 2, "p": 6}).to_string(index=False))


HIPP_CN = CN.neuron[CN.area == "hippocampus"].tolist()
AMYG_CN = CN.neuron[CN.area == "amygdala"].tolist()
READ4 = [("trial", "B1"), ("trial", "B2"), ("code", "B1"), ("code", "B2")]

## Claim fig2b-hippocampus-persistent

> Hippocampal category neurons are persistently active above baseline during maintenance and
> fire more when an image of their preferred category is held in WM.

All correct WM trials, both loads; FR ~ pref with the random intercepts of G. Readings: rows
(a) one per trial or (b) the code's rows, baseline B1 or B2; outliers O-G (main), O-C and none
shown beside.

In [ ]:
R2B = fit_readings(HIPP_CN, lambda d: d[d.acc == 1], "pref", READ4)
show(R2B)

**Result: partially reproduced.** Under every reading with O-G both terms are positive at p < 0.05 and at least one beta is outside tolerance: one row per trial with baseline B1 gives intercept 20.22 (p = 0.00096, outside 17.86 +- 1.79) and preferred 14.11 (p = 0.000001, outside 12.47 +- 1.25); with B2, 22.38 and 11.86 (the preferred beta within tolerance); with the code's rows, 22.98 and 8.82 (B1) or 25.42 and 6.90 (B2). O-G removes 1 or 2 of the 102 neurons. The code's rule (O-C, variant) with one row per trial and B1 gives 19.39 and 13.12, both within tolerance; with B2 it removes the same neurons as O-G.

## Claim fig2c-hippocampus-correct-load

> When the preferred category is held, hippocampal category neurons fire more in correct than
> incorrect trials and more in load 1 than load 3 trials.

Trials holding a preferred picture, correct and incorrect, both loads (one row per trial);
acc (incorrect the reference) and l1 (load 1 against load 3). Readings: (a) main effects, (b)
with their interaction; baseline B1 or B2.

In [ ]:
def pref_trials(d):
    d = d[d.pref == 1].copy()
    d["l1"] = (d.load == 1).astype(int)
    return d

R2C = pd.concat([fit_readings(HIPP_CN, pref_trials, f,
                              [("trial", "B1"), ("trial", "B2")]).assign(model=lab)
                 for lab, f in [("(a) main", "acc + l1"), ("(b) interaction", "acc * l1")]])
show(R2C[["model", "rows", "base", "outliers", "term", "beta", "SE", "p", "neurons", "n"]])
d = pref_trials(wm_table(HIPP_CN, "trial", "B1"))
print("rows by accuracy and load (B1):", d.groupby(["acc", "l1"]).size().to_dict())

**Result: partially reproduced (resting on an assumption).** With main effects only (the adopted reading) and O-G, baseline B1 gives accuracy 18.48 (p = 0.0499) and load 18.64 (p = 0.0003), both positive at p < 0.05 with both betas above tolerance (16.22 +- 1.62 and 13.769 +- 1.377); B2 gives load 12.94 (p = 0.0069, within tolerance) and accuracy 16.46 (p = 0.060), the load effect holding and the accuracy effect positive at p >= 0.05. Both grade the claim partially reproduced. The accuracy p with B1 lies just below 0.05. With the interaction term (reading b) the load main effect is negative and not significant (-14.24, p = 0.56 with B1; -18.66, p = 0.41 with B2), which would grade the claim not reproduced. O-C gives accuracy 18.68 (p = 0.028) and load 13.63 (p = 0.0034) with B1.

## Claim fig2e-amygdala-persistent

> Amygdala category neurons are also persistently active during maintenance and fire more for
> the preferred category.

In [ ]:
R2E = fit_readings(AMYG_CN, lambda d: d[d.acc == 1], "pref", READ4)
show(R2E)

**Result: reproduced (resting on an assumption).** With one row per trial (the adopted reading) and O-G, both terms are positive at p < 0.05 with both betas within tolerance (9.78 +- 0.98 and 6.91 +- 0.69): intercept 10.50 and preferred 6.88 with B1 (p = 0.00046 and 0.000002), 10.06 and 6.55 with B2 (p = 0.000014 and 0.000003). With the code's rows (reading b) the intercept is above tolerance and the preferred beta below it (13.06 and 5.62 with B1, 11.26 and 5.61 with B2), which grades the claim partially reproduced. O-C gives the same as O-G with one row per trial and B1, and 9.34 and 5.89 with B2.

## Claim fig2f-amygdala-load-not-accuracy

> When the preferred category is held, amygdala category neurons fire more in load 1 than load
> 3 trials, but not more in correct than incorrect trials.

In [ ]:
R2F = pd.concat([fit_readings(AMYG_CN, pref_trials, f,
                              [("trial", "B1"), ("trial", "B2")]).assign(model=lab)
                 for lab, f in [("(a) main", "acc + l1"), ("(b) interaction", "acc * l1")]])
show(R2F[["model", "rows", "base", "outliers", "term", "beta", "SE", "p", "neurons", "n"]])

**Result: reproduced (resting on an assumption).** The accuracy effect has p >= 0.05 under every reading and outlier rule (with O-G p from 0.19 to 0.24, betas -6.39 to -7.44; paper -6.13, p = 0.24). Under the adopted reading (main effects, baseline B1) with O-G the load effect is 11.55 (p = 0.00002), within 12.309 +- 1.231. With B2 it is 9.51 (p = 0.00025), below tolerance, and with the interaction term it is not significant (p = 0.70 with B1, 0.82 with B2); these readings grade the claim partially reproduced. O-C gives the same as O-G with B1.

## The table of Figure 3 (NWB_computeGLM.m)

One row per WM picture later shown in the LTM test; rate 0 to 2.5 s after maintenance onset,
normalized to the mean baseline over all the neuron's rows (B, as the code); then rows kept
when the WM trial was correct and the picture is of the preferred category. Confidence is
recoded as the Methods say (sbcat.conf3_code). The session whose ratings are all sure is left
out of these confidence analyses (Results, paragraph 2); the code keeps it, shown as a
variant. Area: hipp = 1 for hippocampus, amygdala the reference.

In [ ]:
NOCONF_SI = [si for si, s in enumerate(S) if s["identifier"] in noconf]


def fig3_full(neurons, value="r_maint"):
    d = wm_table(neurons, "code", "B2", value)
    d["hipp"] = (d.area == "hippocampus").astype(int)
    d["conf3"] = sbcat.conf3_code(d.conf)
    d["si"] = [int(n.split("_")[0]) for n in d.neuron]
    return d


def fig3_select(d, pref=1, keep_noconf=False):
    d = d[(d.acc == 1) & (d.pref == pref) & d.conf.notna()]
    if not keep_noconf:
        d = d[~d.si.isin(NOCONF_SI)]
    return d


ALL_CN = HIPP_CN + AMYG_CN
F3FULL = fig3_full(ALL_CN)
F3 = fig3_select(F3FULL)
print("rows", len(F3), "neurons", F3.neuron.nunique(), "by area:",
      F3.groupby("area").neuron.nunique().to_dict())
print("rows by LTM accuracy and confidence:", F3.groupby(["LTMacc", "conf3"]).size().to_dict())
print("neurons dropped for a zero baseline:", len(ALL_CN) - F3FULL.neuron.nunique())
print("FR (%% change) range: %.1f to %.1f" % (F3.FR.min(), F3.FR.max()))


def fig3_fit(data_full, formula, select=fig3_select, outlier_pool=None, label=""):
    """Fit under O-G (main, claims O), O-C (variant) and none. Both rules take each neuron's
    mean over all its rows of the full table, before trial selection, as NWB_computeGLM.m
    does; outlier_pool is that full table when it is not data_full (outliers found over both
    areas for a model of one area)."""
    d = select(data_full)
    pool = data_full if outlier_pool is None else outlier_pool
    res = []
    for orule in ("O-G", "O-C", "none"):
        bad = (sbcat.og_remove(pool) if orule == "O-G" else
               sbcat.oc_remove(pool) if orule == "O-C" else set())
        dd = d[~d.neuron.isin(bad)]
        m = sbcat.mixedlm(dd, formula)
        for term, r in m.iterrows():
            res.append(dict(reading=label, outliers=orule, term=term, beta=r.beta, SE=r.SE, p=r.p,
                            neurons=m.attrs["neurons"], n=m.attrs["n"], removed=len(bad)))
    return pd.DataFrame(res)


## Claim fig3a-area-interactions

> In a GLM over category neurons of both areas, no main effect of LTM accuracy, confidence or
> area is significant, nor the LTM accuracy by confidence interaction, but every interaction
> with area is, so the relation of maintenance activity to later memory differs between
> hippocampus and amygdala.

FR ~ LTMacc * conf3 * hipp. Expected: conf3:hipp 16.88 +- 1.69, LTMacc:hipp 59.78 +- 5.98,
LTMacc:conf3:hipp -28.64 +- 2.86.

In [ ]:
R3A = pd.concat([
    fig3_fit(F3FULL, "LTMacc * conf3 * hipp", label="main"),
    fig3_fit(F3FULL, "LTMacc * conf3 * hipp",
             select=lambda d: fig3_select(d, keep_noconf=True), label="with the all-sure session")])
show(R3A)

**Result: partially reproduced.** With O-G (5 neurons removed, 313 remain) the three interactions with area are significant with the paper's signs (LTM accuracy x area 70.57, p = 0.0011; confidence x area 19.12, p = 0.019; three-way -32.52, p = 0.0017) and no main effect or the accuracy x confidence term is (smallest p 0.24), but all three betas are larger than the paper's and outside tolerance (59.78 +- 5.98, 16.88 +- 1.69, -28.64 +- 2.86). With the all-sure session kept, as the code does, 73.60, 20.23 and -33.74. The code's rule (O-C, variant, 8 removed) gives 56.61, 16.60 and -27.83, all within tolerance, and with the all-sure session kept 59.96, 17.84 and -29.16.

## Claim fig3b-hippocampus-predicts-ltm

> In the hippocampus, maintenance activity of category neurons for the preferred category
> predicts later LTM accuracy and confidence, with an interaction between the two, so activity
> is higher for pictures later remembered than forgotten.

FR ~ LTMacc * conf3 on hippocampal neurons; outliers found over both areas (a) or within the
hippocampus (b). Expected: conf3 20.30 +- 2.03, LTMacc 67.90 +- 6.79, LTMacc:conf3 -30.80 +-
3.08.

In [ ]:
def area_sel(area, keep_noconf=False):
    return lambda d: fig3_select(d[d.area == area], keep_noconf=keep_noconf)

H3FULL = F3FULL[F3FULL.area == "hippocampus"]
A3FULL = F3FULL[F3FULL.area == "amygdala"]
R3B = pd.concat([
    fig3_fit(H3FULL, "LTMacc * conf3", select=area_sel("hippocampus"),
             outlier_pool=F3FULL, label="(a) outliers over both areas"),
    fig3_fit(H3FULL, "LTMacc * conf3", select=area_sel("hippocampus"),
             label="(b) outliers within the area"),
    fig3_fit(H3FULL, "LTMacc * conf3",
             select=area_sel("hippocampus", True), label="(b) with the all-sure session")])
show(R3B)

**Result: reproduced.** With O-G, whether outliers are found over both areas (5 removed) or within the hippocampus (2 removed), the same 98 hippocampal neurons remain, and confidence 21.25 (p = 0.0045), LTM accuracy 67.37 (p = 0.00085) and their interaction -30.69 (p = 0.0014) are significant with the paper's signs and within tolerance (20.30 +- 2.03, 67.90 +- 6.79, -30.80 +- 3.08). With the all-sure session kept, 21.51, 68.18 and -31.05. The code's rule (O-C) gives the same within the area; over both areas it gives 20.25, 57.52 and -27.98, LTM accuracy below tolerance.

## Claim fig3b-amygdala-no-prediction

> In the amygdala, maintenance activity of category neurons does not predict later LTM
> accuracy or confidence.

In [ ]:
R3BA = pd.concat([
    fig3_fit(A3FULL, "LTMacc * conf3", select=area_sel("amygdala"),
             outlier_pool=F3FULL, label="(a) outliers over both areas"),
    fig3_fit(A3FULL, "LTMacc * conf3", select=area_sel("amygdala"),
             label="(b) outliers within the area"),
    fig3_fit(A3FULL, "LTMacc * conf3",
             select=area_sel("amygdala", True), label="(b) with the all-sure session")])
show(R3BA)

**Result: reproduced.** No effect reaches p < 0.05 in the amygdala under any reading or outlier rule (smallest p 0.43). With O-G and outliers over both areas, confidence 1.81 (p = 0.68), LTM accuracy -4.08 (p = 0.71), interaction 2.28 (p = 0.67).

## Claims fig3c-hippocampus-high-confidence and fig3d-amygdala-no-difference

> Across hippocampal category neurons, maintenance firing for the preferred category is higher
> for pictures later remembered than forgotten with high confidence, but not with low
> confidence.

> Across amygdala category neurons, maintenance firing does not differ between later
> remembered and forgotten pictures, with high or low confidence.

Rows of the Figure 3 table (correct WM trials, preferred pictures later tested, the all-sure
session left out). High: remembered rating 6 against forgotten rating 1; low: remembered 4 or 5
against forgotten 2 or 3. Per neuron the mean rate per condition; neurons without rows or
without spikes in a condition, with a zero baseline, or outside mean +- 3 SD of the neuron and
condition values of the test (O-M) are removed; paired permutation t-test (P). Readings: (a)
normalized rate, (b) spikes per second.

In [ ]:
CONDS = {"high": ([6], [1]), "low": ([4, 5], [2, 3])}


def remembered_forgotten(area, level, value, rng, data=None):
    d = F3 if data is None else data
    d = d[d.area == area]
    rem, forg = CONDS[level]
    col = "FR" if value == "norm" else "r_maint"
    a = d[d.conf.isin(rem)].groupby("neuron").agg(v=(col, "mean"), c=("c_maint", "sum"))
    b = d[d.conf.isin(forg)].groupby("neuron").agg(v=(col, "mean"), c=("c_maint", "sum"))
    j = a.join(b, lsuffix="_r", rsuffix="_f", how="inner")
    n_both = len(j)
    j = j[(j.c_r > 0) & (j.c_f > 0)]
    n_spk = len(j)
    vals = j[["v_r", "v_f"]].values
    out = np.any(np.abs(vals - vals.mean()) > 3 * vals.std(ddof=1), axis=1)
    j = j[~out]
    t, p, df = sbcat.perm_ttest_paired(j.v_r, j.v_f, rng)
    return dict(area=area, level=level, value=value, with_both=n_both, with_spikes=n_spk,
                n=len(j), mean_rem=j.v_r.mean(), mean_forg=j.v_f.mean(),
                diff=(j.v_r - j.v_f).mean(), t=t, df=df, p=p), j.index.tolist()


rng = np.random.default_rng(301)
R3CD, KEEP3C = [], {}
for area in ("hippocampus", "amygdala"):
    for level in ("high", "low"):
        for value in ("norm", "Hz"):
            r, kept = remembered_forgotten(area, level, value, rng)
            R3CD.append(r)
            KEEP3C[(area, level, value)] = kept
R3CD = pd.DataFrame(R3CD)
F3ALL = fig3_select(F3FULL, keep_noconf=True)
R3CDV = pd.DataFrame([remembered_forgotten(a, l, v, np.random.default_rng(303), data=F3ALL)[0]
                      for a in ("hippocampus", "amygdala") for l in ("high", "low")
                      for v in ("norm", "Hz")])
print("category neurons with rows in the Figure 3 table: hippocampus",
      F3[F3.area == "hippocampus"].neuron.nunique(), "amygdala", F3[F3.area == "amygdala"].neuron.nunique())

In [ ]:
print(R3CD[R3CD.area == "hippocampus"].round(4).to_string(index=False))
print("variant with the all-sure session (as the paper's code keeps it):")
print(R3CDV[R3CDV.area == "hippocampus"].round(4).to_string(index=False))
print("paper: high t(82) = 2.16, p = 0.028; low t(68) = -1.38, p = 0.17")

**Result: not reproduced.** In the hippocampus, remembered and forgotten do not differ with high confidence under either reading (normalized: t(68) = 1.17, p = 0.26; spikes per second: t(70) = 1.79, p = 0.078), nor with low confidence (p = 0.30 and 0.10). Keeping the all-sure session, as the code does, gives p = 0.33 and 0.054. 69 to 73 neurons enter the high-confidence test against 83 in the paper.

In [ ]:
print(R3CD[R3CD.area == "amygdala"].round(4).to_string(index=False))
print("variant with the all-sure session:")
print(R3CDV[R3CDV.area == "amygdala"].round(4).to_string(index=False))
print("paper: high t(158) = 0.84, p = 0.41; low t(152) = 1.15, p = 0.26")

**Result: reproduced.** In the amygdala no comparison has p < 0.05 (high p = 0.85 and 0.64, low p = 0.30 and 0.51; 146 to 147 neurons).

## Claims fig3e-early-delay-cluster and fig3f-amygdala-no-cluster

> In hippocampal category neurons, firing differs between pictures later remembered and
> forgotten with high confidence early in the maintenance period, from 0 to 650 ms.

> In amygdala category neurons, no period of the maintenance period shows a significant
> difference between later remembered and forgotten pictures.

Neurons and conditions of the high-confidence test of fig3c (and fig3d), per reading of the
rate. Time-resolved rates over 0 to 2.5 s after maintenance onset: (a) 200 ms windows stepped
by 25 ms, centres 0 to 2.5 s (windows reach 100 ms past the period at both ends), or (b) 50 ms
bins, centres 25 ms to 2.475 s. Normalized rates use each neuron's mean baseline rate of the
Figure 3 table. Cluster-based permutation test (sbcat.cluster_test), 10,000 sign flips,
cluster-forming threshold two-sided 0.05, cluster p judged at 0.025.

In [ ]:
BINNINGS = {"(a) 200/25 ms": (np.arange(0, 2.5001, 0.025), 0.2),
            "(b) 50 ms": (np.arange(0.025, 2.5, 0.05), 0.05)}
BASE3 = {n: g.r_bsl.mean() for n, g in F3FULL.groupby("neuron")}
NEURON_SPIKES = {n: S[u.si]["spikes"][u.ui] for n, u in CN.set_index("neuron").iterrows()}


def time_resolved(area, value, binning, rng):
    centres, width = BINNINGS[binning]
    neurons = KEEP3C[(area, "high", value)]
    d = F3[(F3.area == area) & F3.neuron.isin(neurons)]
    A, B = [], []
    for n in neurons:
        g = d[d.neuron == n]
        rates = []
        for confs in ([6], [1]):
            t0 = g.t_maint[g.conf.isin(confs)].values
            r = sbcat.binned_counts(NEURON_SPIKES[n], t0, centres, width).mean(0) / width
            rates.append(sbcat.normalize(r, BASE3[n]) if value == "norm" else r)
        A.append(rates[0])
        B.append(rates[1])
    t, cl = sbcat.cluster_test(np.array(A), np.array(B), rng)
    return centres, t, cl, len(neurons)


R3EF = []
t0 = time.time()
for area in ("hippocampus", "amygdala"):
    for value in ("norm", "Hz"):
        for binning in BINNINGS:
            rng = np.random.default_rng(302)
            centres, t, cl, n = time_resolved(area, value, binning, rng)
            for c in cl:
                R3EF.append(dict(area=area, value=value, binning=binning, neurons=n,
                                 sign="rem > forg" if c["sign"] == 1 else "rem < forg",
                                 start_s=centres[c["start"]], end_s=centres[c["end"]],
                                 cluster_sum=c["sum"], p=c["p"]))
            if not cl:
                R3EF.append(dict(area=area, value=value, binning=binning, neurons=n, sign="none"))
            print(area, value, binning, "neurons", n, "max |t| %.2f" % np.max(np.abs(t)),
                  "clusters", len(cl))
R3EF = pd.DataFrame(R3EF)
print("cluster tests took %.0f s" % (time.time() - t0))

In [ ]:
r = R3EF[R3EF.area == "hippocampus"]
print(r.round(4).to_string(index=False))
print("paper: 0 to 650 ms, cluster p = 0.0042")

**Result: partially reproduced.** Under every reading a cluster with remembered above forgotten has cluster p < 0.025, but it lies from about 0.875 to 1.05 s (p = 0.0001 to 0.0196), not from 0 to 650 ms. Clusters starting at maintenance onset do not reach significance (smallest p 0.061, 0 to 0.125 s, spikes per second, 200 ms windows).

In [ ]:
r = R3EF[R3EF.area == "amygdala"]
print(r.round(4).to_string(index=False))
print("smallest cluster p per reading:")
print(r.groupby(["value", "binning"]).p.min())

**Result: reproduced.** In the amygdala no cluster has p < 0.025 under any reading (smallest cluster p 0.43).

## Claim fig3g-nonpreferred-no-effect

> Maintenance activity of category neurons while non-preferred pictures are held does not
> predict later LTM accuracy or confidence.

The Figure 3a model on non-preferred rows: (a) every non-preferred picture of the code's
table, (b) only pictures of trials that held no preferred picture.

In [ ]:
trial_has_pref = {n: set(NPT[n].trial[NPT[n].pref == 1]) for n in ALL_CN}


def nonpref_b(d):
    d = fig3_select(d, pref=0)
    keep = [t not in trial_has_pref[n] for n, t in zip(d.neuron, d.trial)]
    return d[keep]


R3G = pd.concat([
    fig3_fit(F3FULL, "LTMacc * conf3 * hipp", select=lambda d: fig3_select(d, pref=0),
             label="(a) all non-preferred pictures"),
    fig3_fit(F3FULL, "LTMacc * conf3 * hipp", select=nonpref_b,
             label="(b) trials without a preferred picture")])
show(R3G)
r = R3G[(R3G.term != "Intercept") & (R3G.outliers.isin(["O-G", "O-C"]))]
print("smallest p of a fixed effect other than the intercept, per reading and outlier rule:")
print(r.groupby(["reading", "outliers"]).p.min())

**Result: reproduced.** With O-G no fixed effect other than the intercept has p < 0.05 in either set of rows (smallest p 0.089 for all non-preferred pictures, 0.11 for trials without a preferred picture), nor with O-C (0.067 and 0.11).

## Claim fig3h-encoding-no-effect

> Activity of category neurons during the presentation of the first encoding picture, for
> preferred pictures, does not predict later LTM accuracy or confidence.

Rate 0 to 2 s after the onset of encoding picture 1, normalized as B; correct WM trials;
(a) trials whose first picture is of the preferred category, with that picture's LTM outcome,
or (b) trials holding a preferred picture in any slot, with that picture's LTM outcome.

In [ ]:
E3FULL = fig3_full(ALL_CN, value="r_enc1")
R3H = pd.concat([
    fig3_fit(E3FULL, "LTMacc * conf3 * hipp",
             select=lambda d: fig3_select(d[d.slot == 1]), label="(a) picture 1 preferred"),
    fig3_fit(E3FULL, "LTMacc * conf3 * hipp", select=fig3_select,
             label="(b) preferred picture in any slot")])
show(R3H)
r = R3H[(R3H.term != "Intercept") & (R3H.outliers.isin(["O-G", "O-C"]))]
print("smallest p of a fixed effect other than the intercept, per reading and outlier rule:")
print(r.groupby(["reading", "outliers"]).p.min())

**Result: reproduced.** With O-G no fixed effect other than the intercept has p < 0.05 under either reading (smallest p 0.30 with picture 1 preferred, 0.078 with a preferred picture in any slot), nor with O-C (0.35 and 0.074). No reading shows an effect of LTM accuracy or confidence.

## Claim ms-neuron-counts

> 53 of the 351 hippocampal units (15.10%) are memory-selective, 25 of them (47%) firing more to
> familiar than novel pictures.

In [ ]:
for area in ("hippocampus", "amygdala"):
    k = K[K.area == area]
    print(f"{area}: {int(k.ms.sum())} of {len(k)} MS ({100*k.ms.mean():.2f} %), FS {int((k.ms_type == 'FS').sum())}, "
          f"NS {int((k.ms_type == 'NS').sum())}")
print("expected hippocampus 53 +- 5.3 MS and 25 +- 2.5 FS; amygdala 68 in Table S1 (report)")
print("MS neurons by patient (hippocampus):", K[(K.area == "hippocampus") & K.ms].patient.value_counts().to_dict())
# Table S1 per-session category and MS counts, as a report for table-s1-counts
cnt = K.groupby(["session", "area"])[["cat", "ms"]].sum().unstack(fill_value=0)
rep = TS1[["label", "session", "h_cat", "a_cat", "h_ms", "a_ms"]].copy()
rep["h_cat_here"] = rep.session.map(cnt[("cat", "hippocampus")]).fillna(0).astype(int)
rep["a_cat_here"] = rep.session.map(cnt[("cat", "amygdala")]).fillna(0).astype(int)
rep["h_ms_here"] = rep.session.map(cnt[("ms", "hippocampus")]).fillna(0).astype(int)
rep["a_ms_here"] = rep.session.map(cnt[("ms", "amygdala")]).fillna(0).astype(int)
for a in ("h_cat", "a_cat", "h_ms", "a_ms"):
    print(f"Table S1 {a}: sessions equal {int((rep[a] == rep[a + '_here']).sum())} of {len(rep)}, "
          f"total {rep[a].sum()} in the table, {rep[a + '_here'].sum()} here")

**Result: reproduced.** 53 of 351 hippocampal units are memory-selective (15.10%), 26 of them familiarity-selective, within tolerance of 53 and 25.

## Claim fig4b-ms-confidence-auc

> Hippocampal memory-selective neurons separate old from new pictures better for
> high-confidence than for low-confidence decisions (higher ROC AUC), so they carry a
> memory-strength signal.

Spike counts 0.2 to 1.2 s after LTM picture onset. FS neurons: all true negatives against
true positives rated 6 (high) and rated 4 or 5 (low); NS neurons: all true positives against
true negatives rated 1 (high) and 2 or 3 (low); the AUC is oriented so that the preferred
response above 0.5 is selectivity (sbcat.roc_auc). The all-sure session is left out. Neurons
kept: (a) at least ten correct novel and ten correct familiar trials, (b) at least ten trials in
each group compared; a neuron needs trials in both groups of each comparison; then O-M on the
two AUC values.

In [ ]:
MSH = K[(K.area == "hippocampus") & K.ms].copy()
rows = []
for _, u in MSH.iterrows():
    s = S[u.si]
    ltm = s["ltm"]
    c = sbcat.count(s["spikes"][u.ui], ltm.timestamps_PicOnset.values, 0.2, 1.2)
    conf, old = ltm.confidence.values, ltm.new_old.values == 1
    tp, tn = old & (conf >= 4), ~old & (conf <= 3)
    if u.ms_type == "FS":
        fixed, high, low = c[tn], c[tp & (conf == 6)], c[tp & ((conf == 4) | (conf == 5))]
        auc = lambda grp: sbcat.roc_auc(grp, fixed)
    else:
        fixed, high, low = c[tp], c[tn & (conf == 1)], c[tn & ((conf == 2) | (conf == 3))]
        auc = lambda grp: sbcat.roc_auc(grp, fixed)
    ok = len(high) > 0 and len(low) > 0 and np.r_[fixed, high, low].max() > 0
    rows.append(dict(neuron=f"{u.si}_{u.ui}", session=u.session, type=u.ms_type,
                     n_tp=int(tp.sum()), n_tn=int(tn.sum()), n_fixed=len(fixed), n_high=len(high),
                     n_low=len(low), auc_high=auc(high) if ok else np.nan,
                     auc_low=auc(low) if ok else np.nan, noconf=u.si in NOCONF_SI))
AUC = pd.DataFrame(rows)
print("hippocampal MS neurons:", len(AUC), " in the all-sure session:", int(AUC.noconf.sum()),
      " without trials in a compared group:", int(AUC.auc_high.isna().sum()))
rng = np.random.default_rng(401)
R4B = []
for lab, sel in [("(a) >= 10 TN and >= 10 TP", lambda a: (a.n_tp >= 10) & (a.n_tn >= 10)),
                 ("(b) >= 10 in each group compared",
                  lambda a: (a.n_fixed >= 10) & (a.n_high >= 10) & (a.n_low >= 10))]:
    a = AUC[sel(AUC) & ~AUC.noconf & AUC.auc_high.notna()]
    v = a[["auc_high", "auc_low"]].values
    out = np.any(np.abs(v - v.mean()) > 3 * v.std(ddof=1), axis=1)
    a = a[~out]
    t, p, df = sbcat.perm_ttest_paired(a.auc_high, a.auc_low, rng)
    R4B.append(dict(reading=lab, neurons=len(a), outliers=int(out.sum()), FS=int((a.type == "FS").sum()),
                    auc_high=a.auc_high.mean(), auc_low=a.auc_low.mean(), t=t, df=df, p=p))
print(pd.DataFrame(R4B).round(4).to_string(index=False))
print("paper: t(40) = 2.89, p = 0.0039, 41 neurons")
print("AUC range: %.3f to %.3f" % (np.nanmin(AUC[["auc_high", "auc_low"]].values),
                                   np.nanmax(AUC[["auc_high", "auc_low"]].values)))

**Result: reproduced.** The AUC is higher for high than low confidence under both readings: 0.5975 against 0.5577 (t(48) = 3.87, p = 0.0005, 49 neurons) and 0.5994 against 0.5621 (t(39) = 3.55, p = 0.0009, 40 neurons; paper 41).

## Claim fig4c-overlap-chance

> 20 hippocampal neurons are both memory-selective and category neurons, an overlap no greater
> than chance.

In [ ]:
rng = np.random.default_rng(402)
H = K[K.area == "hippocampus"]
obs = int((H.cat & H.ms).sum())
ncat, ms = int(H.cat.sum()), H.ms.values
null = np.array([ms[rng.choice(len(H), ncat, replace=False)].sum() for _ in range(1000)])
p = np.mean(null >= obs)
print(f"overlap {obs} (expected 20 +- 2) of {int(ms.sum())} MS and {ncat} category neurons; "
      f"null mean {null.mean():.2f}, range {null.min()} to {null.max()}; p = {p:.3f} (paper 0.11)")

**Result: reproduced.** 19 neurons are both memory-selective and category neurons (expected 20 +- 2); the null overlap has mean 15.55 and p = 0.179, not below 0.05.

## Claim fig4d-category-no-memory-signal

> During LTM retrieval, hippocampal category neurons code neither familiarity nor confidence
> for pictures of their preferred category.

The new pictures' categories are not in the dataset (see below); the tests here use the
categories inferred from the ID blocks and are shown as a variant only.

LTM trials showing a picture of the neuron's preferred category; rate 0.2 to 1.2 s after
picture onset. GLM on correct trials (all-sure session left out): FR ~ conf * old with the
random intercepts of G. Readings: (D1) confidence coding (a) the code's or (b) certainty;
(D2) rate (a) spikes per second or (b) normalized to the neuron's mean WM baseline (B1). The
t-test: per neuron the mean rate of preferred old against preferred new pictures, (D3) (a)
correct trials or (b) all trials, under both D2 readings, after O-M; paired permutation t-test.

The category of a new LTM picture is not in the dataset. New pictures have IDs 1001 to 1200
(old ones keep their WM IDs, whose leading digit is the category), and their images in the
files are named only by ID. The cell below asks whether the IDs are in category blocks of 40
(1001 to 1040 the first category, and so on), from the category neurons' responses: for each
category neuron, the block with the highest mean response to new pictures is compared with
its preferred category, against the same for old pictures and for a grouping by ID modulo 5.
This is an inference from the neural data, not a label, so the claim is not decided on it.

In [ ]:
def new_cat_block(ids):
    return (np.asarray(ids, int) - 1001) // 40 + 1


match = {"old pictures, category from the ID": 0, "new pictures, blocks of 40": 0,
         "new pictures, ID modulo 5": 0}
for _, u in CN.iterrows():
    s = S[u.si]
    l = s["ltm"]
    c = sbcat.count(NEURON_SPIKES[u.neuron], l.timestamps_PicOnset.values, 0.2, 1.2)
    new = l.new_old.values == 0
    ids = l.PicIDs.values.astype(int)
    for lab, sel, g in [("old pictures, category from the ID", ~new, ids // 100),
                        ("new pictures, blocks of 40", new, new_cat_block(ids)),
                        ("new pictures, ID modulo 5", new, (ids - 1001) % 5 + 1)]:
        m = [c[sel & (g == k)].mean() if np.any(sel & (g == k)) else -1 for k in range(1, 6)]
        match[lab] += int(np.argmax(m) + 1 == u.pref)
for k, v in match.items():
    print(f"{k}: the best group is the preferred category for {v} of {len(CN)} category neurons "
          f"({100*v/len(CN):.1f} %; chance 20 %)")
print("new picture IDs:", int(min(s["ltm"].PicIDs[s["ltm"].new_old == 0].min() for s in S)), "to",
      int(max(s["ltm"].PicIDs[s["ltm"].new_old == 0].max() for s in S)))

In [ ]:
rows = []
for n in HIPP_CN:
    u = CN.set_index("neuron").loc[n]
    s = S[u.si]
    ltm = s["ltm"]
    ids = ltm.PicIDs.values.astype(int)
    cat_ltm = np.where(ltm.new_old.values == 1, ids // 100, new_cat_block(ids))
    sel = cat_ltm == u.pref
    c = sbcat.count(NEURON_SPIKES[n], ltm.timestamps_PicOnset.values[sel], 0.2, 1.2)
    b = trial_rows(NPT[n]).r_bsl.mean()
    rows.append(pd.DataFrame(dict(neuron=n, patient=u.patient, si=u.si, c=c, Hz=c / 1.0,
                                  norm=sbcat.normalize(c / 1.0, b) if b > 0 else np.nan,
                                  conf=ltm.confidence.values[sel], old=ltm.new_old.values[sel],
                                  acc=ltm.response_accuracy.values[sel])))
L4 = pd.concat(rows, ignore_index=True)
L4["conf_code"] = sbcat.conf3_code(L4.conf)
L4["conf_cert"] = sbcat.conf3_certainty(L4.conf)
print("rows", len(L4), "neurons", L4.neuron.nunique(), " correct", int(L4.acc.sum()),
      " neurons with a zero WM baseline:", L4[L4.norm.isna()].neuron.nunique())

In [ ]:
res = []
for d1 in ("conf_code", "conf_cert"):
    for d2 in ("Hz", "norm"):
        d = L4[(L4.acc == 1) & ~L4.si.isin(NOCONF_SI) & L4[d2].notna()].copy()
        d["FR"], d["confv"], d["old"] = d[d2], d[d1], d.old.astype(int)
        for orule in ("O-M", "O-C", "none"):
            bad = (sbcat.om_remove(d, ["old"], zero_col="c") if orule == "O-M" else
                   sbcat.oc_remove(d) if orule == "O-C" else set())
            m = sbcat.mixedlm(d[~d.neuron.isin(bad)], "confv * old")
            for term, r in m.iterrows():
                res.append(dict(D1=d1, D2=d2, outliers=orule, term=term, beta=r.beta, p=r.p,
                                neurons=m.attrs["neurons"], n=m.attrs["n"]))
R4D = pd.DataFrame(res)
print(R4D.round(4).to_string(index=False))
print("smallest p of a non-intercept effect per reading (O-M):")
print(R4D[(R4D.term != "Intercept") & (R4D.outliers == "O-M")].groupby(["D1", "D2"]).p.min())
rng = np.random.default_rng(403)
res = []
for d3 in ("correct", "all"):
    for d2 in ("Hz", "norm"):
        d = L4[L4[d2].notna()]
        if d3 == "correct":
            d = d[d.acc == 1]
        a = d[d.old == 1].groupby("neuron").agg(v=(d2, "mean"), c=("c", "sum"))
        b = d[d.old == 0].groupby("neuron").agg(v=(d2, "mean"), c=("c", "sum"))
        j = a.join(b, lsuffix="_o", rsuffix="_n", how="inner")
        j = j[(j.c_o > 0) & (j.c_n > 0)]
        v = j[["v_o", "v_n"]].values
        j = j[~np.any(np.abs(v - v.mean()) > 3 * v.std(ddof=1), axis=1)]
        t, p, df = sbcat.perm_ttest_paired(j.v_o, j.v_n, rng)
        res.append(dict(D3=d3, D2=d2, neurons=len(j), old=j.v_o.mean(), new=j.v_n.mean(), t=t, df=df, p=p))
print(pd.DataFrame(res).round(4).to_string(index=False))
print("paper: GLM confidence p = 0.28, familiarity p = 0.71, interaction p = 0.57; t(94) = -0.28, p = 0.78")

**Result: blocked (metadata insufficient).** The category of the new LTM pictures is not in the dataset, so preferred new pictures cannot be selected. Under categories inferred from ID blocks of 40, shown as a variant that decides nothing, no GLM effect (smallest p 0.079) and no old against new t-test (smallest p 0.22) reaches p < 0.05, as the paper reports.

## Claims fig4e-fs-strong-maintenance and fig4f-ns-no-effect

> Familiarity-selective hippocampal neurons respond more during LTM retrieval to familiar
> pictures that were held with strong, compared with weak, maintenance activity of a
> simultaneously recorded category neuron.

> The response of novelty-selective hippocampal neurons to familiar pictures does not depend on
> whether the paired category neuron's maintenance activity was strong or weak.

Pairs of a hippocampal MS neuron and a hippocampal category neuron of the same session. Per
category neuron, its maintenance rates in correct WM trials holding a preferred picture later
shown in the LTM test, median-split within load 1 and within load 3 (a rate equal to the median
is weak). The MS neuron's rate 0.2 to 1.2 s after onset of those pictures in the LTM test.
Readings: (E1) the random intercept within patient is (a) the MS neuron or (b) the pair; (E2)
(a) all LTM trials of those pictures or (b) those answered old; (E3) a neuron both MS and
category (a) paired with itself or (b) not; (E4) rate (a) spikes per second or (b) normalized to
the MS neuron's mean WM baseline.

In [ ]:
MSN = K[(K.area == "hippocampus") & K.ms].copy()
MSN["neuron"] = [f"{a}_{b}" for a, b in zip(MSN.si, MSN.ui)]
prow = []
for _, m in MSN.iterrows():
    sp = S[m.si]["spikes"][m.ui]
    trs = sbcat.add_counts(PT[m.si], sp)
    mbase = trs.groupby("trial").r_bsl.first().mean()
    for n in CN.neuron[(CN.area == "hippocampus") & (CN.si == m.si)]:
        pt = code_rows(NPT[n])
        pt = pt[(pt.acc == 1) & (pt.pref == 1)].copy()
        if len(pt) == 0:
            continue
        med = pt.groupby("load").r_maint.transform("median")
        pt["strong"] = (pt.r_maint > med).astype(int)
        c = sbcat.count(sp, pt.t_ltm.values, 0.2, 1.2)
        prow.append(pd.DataFrame(dict(ms=m.neuron, cat=n, pair=f"{m.neuron}|{n}", patient=m.patient,
                                      type=m.ms_type, self=m.neuron == n, strong=pt.strong.values,
                                      load=pt.load.values, LTMacc=pt.LTMacc.values, c=c, Hz=c / 1.0,
                                      norm=sbcat.normalize(c / 1.0, mbase) if mbase > 0 else np.nan)))
P4 = pd.concat(prow, ignore_index=True)
pairs = P4.drop_duplicates("pair")
print("pairs:", len(pairs), " FS:", int((pairs.type == "FS").sum()), " NS:", int((pairs.type == "NS").sum()),
      " self-pairs:", int(pairs.self.sum()), "(FS", int((pairs.self & (pairs.type == "FS")).sum()), ")")
print("without self-pairs:", int((~pairs.self).sum()), " FS:", int(((pairs.type == "FS") & ~pairs.self).sum()),
      " NS:", int(((pairs.type == "NS") & ~pairs.self).sum()))
print("rows per pair: %d to %d; strong share %.3f" % (P4.groupby("pair").size().min(),
      P4.groupby("pair").size().max(), P4.strong.mean()))
print("expected: 198 +- 20 pairs, 76 +- 8 FS pairs (paper: 122 NS)")


def pair_tests(typ, rng):
    glm, tt = [], []
    for e2 in ("all", "answered old"):
        for e3 in ("with self", "without self"):
            for e4 in ("Hz", "norm"):
                d = P4[(P4.type == typ) & P4[e4].notna()]
                if e2 == "answered old":
                    d = d[d.LTMacc == 1]
                if e3 == "without self":
                    d = d[~d.self]
                d = d.assign(FR=d[e4])
                for e1 in ("MS neuron", "pair"):
                    dd = d.assign(neuron=d.ms if e1 == "MS neuron" else d.pair)
                    bad = sbcat.om_remove(dd.assign(neuron=d.pair), ["strong"], zero_col="c")
                    m = sbcat.mixedlm(dd[~d.pair.isin(bad)], "strong")
                    glm.append(dict(E1=e1, E2=e2, E3=e3, E4=e4, beta=m.loc["strong", "beta"],
                                    p=m.loc["strong", "p"], pairs=d.pair[~d.pair.isin(bad)].nunique(),
                                    n=m.attrs["n"]))
                a = d[d.strong == 1].groupby("pair").agg(v=("FR", "mean"), c=("c", "sum"))
                b = d[d.strong == 0].groupby("pair").agg(v=("FR", "mean"), c=("c", "sum"))
                j = a.join(b, lsuffix="_s", rsuffix="_w", how="inner")
                j = j[(j.c_s > 0) & (j.c_w > 0)]
                v = j[["v_s", "v_w"]].values
                j = j[~np.any(np.abs(v - v.mean()) > 3 * v.std(ddof=1), axis=1)]
                t, p, df = sbcat.perm_ttest_paired(j.v_s, j.v_w, rng)
                tt.append(dict(E2=e2, E3=e3, E4=e4, pairs=len(j), strong=j.v_s.mean(), weak=j.v_w.mean(),
                               t=t, df=df, p=p))
    return pd.DataFrame(glm), pd.DataFrame(tt)


G4E, T4E = pair_tests("FS", np.random.default_rng(404))
G4F, T4F = pair_tests("NS", np.random.default_rng(405))

In [ ]:
print(G4E.round(4).to_string(index=False))
print(T4E.round(4).to_string(index=False))
print("paper: beta = 14.63, p = 0.00139; t(75) = 3.43, p = 0.0003")
print("GLM readings with strong > weak at p < 0.05:", int(((G4E.beta > 0) & (G4E.p < 0.05)).sum()), "of", len(G4E))
print("t-test readings with strong > weak at p < 0.05:", int(((T4E.t > 0) & (T4E.p < 0.05)).sum()), "of", len(T4E))

**Result: not reproduced.** In familiarity-selective pairs, MS responses after strong maintenance are not higher than after weak under any reading: the GLM betas are negative (-0.14 to -0.19 Hz, -6.5 to -9.4 normalized; p 0.059 to 0.45), and so are the paired t-tests (t from -1.00 to -1.60, p 0.11 to 0.33). There are 197 pairs (expected 198 +- 20) and 84 FS pairs (expected 76 +- 8).

In [ ]:
print(G4F.round(4).to_string(index=False))
print(T4F.round(4).to_string(index=False))
print("paper: beta = 0.09, p = 0.98; t(121) = -0.48, p = 0.64")
print("GLM readings with p < 0.05:", int((G4F.p < 0.05).sum()), "of", len(G4F),
      "; t-test readings with p < 0.05:", int((T4F.p < 0.05).sum()), "of", len(T4F))

**Result: reproduced (resting on an assumption).** Under the adopted reading (all trials of the old pictures, self-pairs kept, spikes per second) neither test finds an effect in novelty-selective pairs (GLM p = 0.90 and 0.85; t(106) = -1.03, p = 0.31). No GLM reading reaches p < 0.05; one t-test reading does (pictures answered old, self-pairs dropped, spikes per second: t(81) = 2.04, p = 0.040).

## Summary

| Claim | Outcome |
|---|---|
| sessions-and-neurons | reproduced |
| table-s1-counts | partially reproduced (42 of 46 sessions) |
| behaviour-overall-accuracy | reproduced |
| fig1c-wm-accuracy-load | reproduced |
| fig1d-wm-rt-load | partially reproduced (p = 0.0003, not below 0.0001) |
| fig1e-ltm-confidence-accuracy | reproduced |
| ltm-rt-confidence | reproduced |
| fig1f-ltm-load | reproduced |
| fig1g-ltm-probe | reproduced |
| category-neuron-counts | reproduced |
| fig2b-hippocampus-persistent | partially reproduced |
| fig2c-hippocampus-correct-load | partially reproduced (assumed) |
| fig2e-amygdala-persistent | reproduced (assumed) |
| fig2f-amygdala-load-not-accuracy | reproduced (assumed) |
| fig3a-area-interactions | partially reproduced |
| fig3b-hippocampus-predicts-ltm | reproduced |
| fig3b-amygdala-no-prediction | reproduced |
| fig3c-hippocampus-high-confidence | not reproduced |
| fig3d-amygdala-no-difference | reproduced |
| fig3e-early-delay-cluster | partially reproduced |
| fig3f-amygdala-no-cluster | reproduced |
| fig3g-nonpreferred-no-effect | reproduced |
| fig3h-encoding-no-effect | reproduced |
| ms-neuron-counts | reproduced |
| fig4b-ms-confidence-auc | reproduced |
| fig4c-overlap-chance | reproduced |
| fig4d-category-no-memory-signal | blocked, metadata insufficient |
| fig4e-fs-strong-maintenance | not reproduced |
| fig4f-ns-no-effect | reproduced (assumed) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. For the GLMs of Figures 2 and 3 the outlier rule is
O-G, as the claims file has it after a reader's answer (questions/001187.yaml): the code's
neuron means (NWB_computeGLM.m), with the Methods' "±3 SD from the mean"; the code's own rule
(O-C, above 3 SD without centring) is shown as a variant, and gives the paper's Figure 3a betas
within tolerance where O-G does not. The permutation t-tests keep the Methods rule (O-M). For
Figure 2, which the code does not cover, one row per trial is adopted (the text's description)
and the baseline is the mean over all WM trials of the neuron. The session whose ratings are
all sure is left out of confidence analyses, as the paper says; the code keeps it, shown as a
variant. Mixed models are fitted by maximum likelihood in statsmodels, not MATLAB's fitglme.
Permutation p-values follow statcond's formula.

What was found about the dataset. The 46 sessions, 950 units and the 351 and 532 units after
the rate filter match the paper. Table S1 lists 23 more units in four sessions. All files give
Cedars-Sinai as the institution, though the identifiers (TWH, JHU) and the paper place 20
sessions at Toronto Western and Johns Hopkins. Eight sessions have fewer than 400 LTM trials
(161 to 399) and one has 139 WM trials. One LTM trial has confidence 7, outside the documented
1 to 6. The new LTM pictures (IDs 1001 to 1200) carry no category; the neural responses
suggest blocks of 40 per category. Five units have spike times out of order in the files
(sorted here before analysis). The confidence coding (1 to 3 new, 4 to 6 old) is not
documented but agrees with the accuracy column in every trial.

In [ ]:
print("notebook run time %.1f min" % ((time.time() - T0) / 60))